<a href="https://colab.research.google.com/github/Nxpze/essencial-project1/blob/main/essencial_project1_Clinic_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# INCORRECT
"ประดิษฐ์", "ประพันธ์" "ประทีป"

# CORRECT
"ประดิษฐ์", "ประพันธ์", "ประทีป"

('ประดิษฐ์', 'ประพันธ์', 'ประทีป')

In [6]:
# INCORRECT (Indentation error)
for q in queue_list:
    q.check_waiting_status(current_sim_time)

    print(f"...")

# The block below is indented unexpectedly:
    if q.alert_status == "ต้องรีเช็ค":
       condition = "อาการแย่ลง" if q.patient_id == "P004" else "อาการคงเดิม"
       q.nurse_recheck(condition)
       print(...)

...
...
...
Ellipsis
...
Ellipsis


In [7]:
from datetime import datetime, timedelta, date
import random
import sqlite3
import matplotlib.pyplot as plt
import pandas as pd

random.seed(42)

# 1. CLASS DEFINITIONS & TRIAGE LOGIC

class Patient:
    def __init__(self, patient_id, fullname, dob, phone_number, allergies="ไม่มี", underlying_disease="ไม่มี"):
        self.patient_id = patient_id
        self.fullname = fullname
        self.dob = dob
        self.phone_number = phone_number
        self.allergies = allergies
        self.underlying_disease = underlying_disease
        self.age = self.age_cal()

    def age_cal(self):
        birthdate = datetime.strptime(self.dob, "%Y-%m-%d").date()
        today = datetime.today().date()
        age = today.year - birthdate.year - ((today.month, today.day) < (birthdate.month, birthdate.day))
        return age

class Queue:
    def __init__(self, queue_id, patient_id, queue_number, symptoms, urgency_level, arrival_time):
        self.queue_id = queue_id
        self.patient_id = patient_id
        self.queue_number = queue_number
        self.symptoms = symptoms
        self.urgency_level = urgency_level  # Red, Yellow, Green, White
        self.arrival_time = arrival_time
        self.waiting_time_threshold = self.get_threshold_by_color()
        self.priority_score = self.get_priority_score()
        self.waiting_time = 0
        self.status = "In Queue"
        self.alert_status = "ปกติ"

    def get_threshold_by_color(self):
        thresholds = {"Red": 0, "Yellow": 15, "Green": 30, "White": 45}
        return thresholds.get(self.urgency_level, 30)

    def get_priority_score(self):
        scores = {"Red": 1, "Yellow": 2, "Green": 3, "White": 4}
        return scores.get(self.urgency_level, 99)

    def check_waiting_status(self, current_time):
        elapsed_minutes = (current_time - self.arrival_time).total_seconds() / 60
        self.waiting_time = round(elapsed_minutes, 1)

        if self.waiting_time >= 30 and self.urgency_level != "Red":
            self.alert_status = "ต้องรีเช็ค"

    def nurse_recheck(self, condition):
        if condition == "อาการแย่ลง":
            self.urgency_level = "Red"
            self.priority_score = 1
            self.alert_status = "ดันคิวทันที (Fast-track)"
            self.waiting_time_threshold = 0
        else:
            self.alert_status = "รีเช็คแล้ว อาการคงเดิม"

    def complete_examination(self):
        self.status = "Return Case"


class MedicalRecord:
    DOCTOR = {
        "DOC001": {"doctor_name": "นพ. สมชาย ใจดี", "specialization": "อายุรกรรม"},
        "DOC002": {"doctor_name": "พญ. วิภาดา รักษาดี", "specialization": "กุมารเวชศาสตร์ (หมอเด็ก)"},
        "DOC003": {"doctor_name": "นพ. ธนกฤต เก่งกาจ", "specialization": "ศัลยกรรมกระดูกและข้อ"}
    }
    def __init__(self, record_id, patient_id, doctor_id, record_date, diagnosis, status, prescribed_meds):
        self.record_id = record_id
        self.patient_id = patient_id
        doctor_info = self.DOCTOR.get(doctor_id, {"doctor_name": "ไม่พบข้อมูลแพทย์", "specialization": "ไม่ระบุ"})
        self.doctor_id = doctor_id
        self.doctor_name = doctor_info.get("doctor_name")
        self.doctor_specialization = doctor_info.get("specialization")
        self.record_date = record_date
        self.diagnosis = diagnosis
        self.status = status
        self.prescribed_meds = prescribed_meds


class Bill:
    def __init__(self, bill_id, record_id, patient_id, treatment_fee, medication_fee, status):
        self.bill_id = bill_id
        self.record_id = record_id
        self.patient_id = patient_id
        self.treatment_fee = treatment_fee
        self.medication_fee = medication_fee
        self.total_payment = self.calculate_total_payment()
        self.status = status

    def calculate_total_payment(self):
        return round(self.treatment_fee + self.medication_fee, 2)


# 2. HELPER FUNCTIONS
def sort_queue_by_triage(queue_list):
    return sorted(queue_list, key=lambda q: (q.priority_score, q.arrival_time))


# 3. WORKFLOW RUNNER
patients_data = [
    ("P001", "กิตติ มีสุข", "1995-05-20", "0811112222", "ไม่มี", "เบาหวาน"),
    ("P002", "นภา มั่นคง", "2018-09-12", "0822223333", "ยาพารา", "ไม่มี"),
    ("P003", "สมชาย เจริญ", "1970-01-01", "0833334444", "ไม่มี", "ความดัน"),
    ("P004", "วิภา รุ่งเรือง", "1988-11-30", "0844445555", "อาหารทะเล", "ไม่มี"),
]

patients_dict = {}
for patient_id, name, dob, phone, allergy, disease in patients_data:
    patients_dict[patient_id] = Patient(patient_id, name, dob, phone, allergy, disease)

sample_symptoms = [
    ("P001", "ไอ มีน้ำมูก", "Green"),
    ("P002", "ขอใบรับรองแพทย์", "White"),
    ("P003", "หมดสติ แน่นหน้าอก", "Red"),
    ("P004", "ปวดท้องเกร็ง ไข้สูง", "Yellow"),
]

start_time = datetime(2026, 8, 23, 9, 0, 0)
queue_list = []

print("=== 1. คัดกรองผู้ป่วยและสร้างคิว ===")
for i, (patient_id, symptom, color) in enumerate(sample_symptoms, 1):
    arr_time = start_time + timedelta(minutes=i * 5)
    q = Queue(f"Q{i:03d}", patient_id, f"A-{i:03d}", symptom, color, arr_time)
    queue_list.append(q)
    print(f"คิว {q.queue_number} | รหัสผู้ป่วย: {q.patient_id} ({patients_dict[patient_id].fullname}) | สี: {q.urgency_level} | เวลามาถึง: {q.arrival_time.strftime('%H:%M')}")

print("\n=== 2. จัดเรียงลำดับคิวตามความรุนแรง ===")
queue_list = sort_queue_by_triage(queue_list)
for pos, q in enumerate(queue_list, 1):
    print(f"ลำดับที่ {pos}: คิว {q.queue_number} (เคสสี {q.urgency_level})")

print("\n=== 3. ประเมินเวลารอคอย และ Re-evaluation ===")
current_sim_time = start_time + timedelta(minutes=40)

for q in queue_list:
    q.check_waiting_status(current_sim_time)
    print(f"เวลารอคอยล่าสุด ณ เวลา {current_sim_time.strftime('%H:%M')} น. | "
          f"คิว: {q.queue_number} | เวลาที่รอไปแล้ว: {q.waiting_time} นาที | "
          f"สถานะเตือน: {q.alert_status}")

    if q.alert_status == "ต้องรีเช็ค":
        condition = "อาการแย่ลง" if q.patient_id == "P004" else "อาการคงเดิม"
        q.nurse_recheck(condition)
        print(f"คิว {q.queue_number} ผลรีเช็คพยาบาล: {condition} -> สถานะใหม่: {q.urgency_level} ({q.alert_status})")

queue_list = sort_queue_by_triage(queue_list)

print("\n=== 4. ตรวจรักษาและจบขั้นตอนที่ RETURN CASE ===")
completed_records = []
doctors_keys = ["DOC001", "DOC002", "DOC003"]

for q in queue_list:
    q.complete_examination()
    doc_id = random.choice(doctors_keys)
    rec = MedicalRecord(
        record_id=f"REC-{q.queue_id}",
        patient_id=q.patient_id,
        doctor_id=doc_id,
        record_date=current_sim_time.strftime("%Y-%m-%d"),
        diagnosis=f"วินิจฉัยเคส {q.urgency_level}: {q.symptoms}",
        status=q.status,
        prescribed_meds="รับยาตามอาการ"
    )
    completed_records.append(rec)
    print(f"คิว {q.queue_number} | แพทย์ผู้ตรวจ: {rec.doctor_name} ({rec.doctor_specialization}) | สถานะเคส: {q.status}")

print("\n=== 5. พบแพทย์ / วินิจฉัย ===")
treatment_results = []
for rec in completed_records:
    treatment = rec.prescribed_meds
    result = random.choice(["หายขาด", "ไม่หายขาด"])
    appointment = "ไม่ต้องนัด" if result == "หายขาด" else "return ใบนัด"

    treatment_results.append({
        "record_id": rec.record_id,
        "patient_id": rec.patient_id,
        "treatment": treatment,
        "treatment_result": result,
        "appointment": appointment
    })

    print(f"ผู้ป่วย: {rec.patient_id} | สั่งยา/นัดหัตถการ: {treatment} | ผลการรักษา: {result} | {appointment}")

print("\n=== 6. Bills ===")
completed_bills = []
for i, rec in enumerate(completed_records, 1):
    treatment_fee = random.choice([300, 500, 800, 1000, 1500])
    medication_fee = random.choice([50, 100, 200, 300, 500])

    bill = Bill(
        bill_id=f"BILL-{i:03d}",
        record_id=rec.record_id,
        patient_id=rec.patient_id,
        treatment_fee=treatment_fee,
        medication_fee=medication_fee,
        status="รอชำระ"
    )
    completed_bills.append(bill)
    print(f"ผู้ป่วย: {bill.patient_id} | ค่ารักษา: {bill.treatment_fee} บาท | ค่ายา: {bill.medication_fee} บาท | รวม: {bill.total_payment} บาท")

# 7. GENERATE RANDOM 300 PATIENTS CSV
first_names = [
    "สมชัย", "สมหญิง", "สมศักดิ์", "สมศรี", "สมฤดี",
    "สมหมาย", "สมบัตร", "สมปอง", "สมใจ", "สมนึก",
    "สมสวย", "สมหวัง", "สมภพ", "สมหทัย", "สมควร"
]

last_names = [
    "ประสม", "ประเสริฐ", "ประเสริฐศรี", "ประสงค์", "ประทาน",
    "ประชาชื่น", "ประเสริฐฐาน", "ประจักร", "ประสิทธิ์", "ประสาน",
    "ประชาโชค", "ประมาณตน", "ประดิษฐ์", "ประพันธ์", "ประทีป"  # Fixed missing comma
]

allergies_list = [
    "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี",
    "พาราเซตามอล", "เพนิซิลลิน", "แอสไพริน", "ซัลฟา", "อัลโลพูรินอล", "เฟนิโทอิน", "ไดโคลฟีแนก"
]

diseases_list = [
    "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี", "ไม่มี",
    "เบาหวาน", "ความดันโลหิตสูง", "หอบหืด", "ไขมันในเลือดสูง", "โรคหัวใจ", "หลอดเลือดหัวใจตีบ", "HIV"
]

symptoms_pool = {
    "Red": ["หมดสติ แน่นหน้าอก", "หายใจลำบากขั้นรุนแรง", "ช็อกจากเสียเลือด", "อุบัติเหตุร้ายแรง", "ชักเกร็งไม่รู้สึกตัว"],
    "Yellow": ["ปวดท้องเกร็ง ไข้สูง", "ปวดศีรษะเฉียบพลัน", "แผลฉีกขาดเลือดไหลไม่หยุด", "อาเจียนต่อเนื่อง", "กระดูกแขนผื่นผิดรูป"],
    "Green": ["ไอ มีน้ำมูก", "ปวดศีรษะเล็กน้อย", "ปวดกล้ามเนื้อ", "ผื่นคันตามตัว", "เจ็บคอ ถ่ายเหลว"],
    "White": ["ขอใบรับรองแพทย์", "มาตามนัดฟังผลตรวจ", "ปรึกษาเรื่องสุขภาพทั่วไป", "ขอรับยาเดิม", "ฉีดวัคซีนตามนัด"]
}

patient_data = []
today = datetime.now()

for i in range(1, 301):
    patient_id = f"P{i:03d}"
    fullname = f"{random.choice(first_names)} {random.choice(last_names)}"
    dob_dt = today - timedelta(days=random.randint(5 * 365, 90 * 365))
    dob = dob_dt.strftime("%Y-%m-%d")
    age = today.year - dob_dt.year - ((today.month, today.day) < (dob_dt.month, dob_dt.day))
    phone_number = f"08{random.randint(10000000, 99999999)}"
    allergies = random.choice(allergies_list)
    diseases = random.choice(diseases_list)
    urgency_level = random.choice(list(symptoms_pool.keys()))
    symptom_detail = random.choice(symptoms_pool[urgency_level])

    patient_data.append({
        "รหัสผู้ป่วย": patient_id,
        "ชื่อ - นามสกุล": fullname,
        "วันเกิด": dob,
        "อายุ": age,
        "เบอร์โทรศัพท์": phone_number,
        "ประวัติแพ้ยา": allergies,
        "โรคประจำตัว": diseases,
        "ระดับอาการ": urgency_level,
        "อาการ": symptom_detail,
    })

df_patients = pd.DataFrame(patient_data)
df_patients.to_csv("patients_300.csv", index=False, encoding="utf-8-sig")
print("\nสร้างข้อมูลผู้ป่วยครบ 300 คนเรียบร้อยแล้ว!")
print(df_patients.head(10))

=== 1. คัดกรองผู้ป่วยและสร้างคิว ===
คิว A-001 | รหัสผู้ป่วย: P001 (กิตติ มีสุข) | สี: Green | เวลามาถึง: 09:05
คิว A-002 | รหัสผู้ป่วย: P002 (นภา มั่นคง) | สี: White | เวลามาถึง: 09:10
คิว A-003 | รหัสผู้ป่วย: P003 (สมชาย เจริญ) | สี: Red | เวลามาถึง: 09:15
คิว A-004 | รหัสผู้ป่วย: P004 (วิภา รุ่งเรือง) | สี: Yellow | เวลามาถึง: 09:20

=== 2. จัดเรียงลำดับคิวตามความรุนแรง ===
ลำดับที่ 1: คิว A-003 (เคสสี Red)
ลำดับที่ 2: คิว A-004 (เคสสี Yellow)
ลำดับที่ 3: คิว A-001 (เคสสี Green)
ลำดับที่ 4: คิว A-002 (เคสสี White)

=== 3. ประเมินเวลารอคอย และ Re-evaluation ===
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-003 | เวลาที่รอไปแล้ว: 25.0 นาที | สถานะเตือน: ปกติ
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-004 | เวลาที่รอไปแล้ว: 20.0 นาที | สถานะเตือน: ปกติ
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-001 | เวลาที่รอไปแล้ว: 35.0 นาที | สถานะเตือน: ต้องรีเช็ค
คิว A-001 ผลรีเช็คพยาบาล: อาการคงเดิม -> สถานะใหม่: Green (รีเช็คแล้ว อาการคงเดิม)
เวลารอคอยล่าสุด ณ เวลา 09:40 น. | คิว: A-002 | เวลาที่รอไปแล้ว